In [1]:
from pathlib import Path
from urllib.request import urlopen
import json

# 1. Khai báo đường dẫn dự án
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "ACME_Insurance"
RESULTS_DIR = PROJECT_DIR / "results" / "llm"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Cấu hình Ollama và mô hình
OLLAMA_URL = "http://127.0.0.1:11434"
MODEL_NAME = "qwen2.5-coder:3b"

# 3. Kiểm tra kết nối Ollama
try:
    with urlopen(f"{OLLAMA_URL}/api/tags", timeout=5) as response:
        data = json.load(response)

    ollama_connected = True
    installed_models = [
        model["name"] for model in data.get("models", [])
    ]

except Exception as e:
    ollama_connected = False
    installed_models = []
    print("Connection error:", e)

# 4. Hiển thị trạng thái môi trường
print("\n=== LOCAL LLM ENVIRONMENT ===")
print("Project directory:", PROJECT_DIR)
print("Model:", MODEL_NAME)
print("Ollama connected:", ollama_connected)
print("Installed models:", installed_models)
print("Results directory:", RESULTS_DIR)


=== LOCAL LLM ENVIRONMENT ===
Project directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Model: qwen2.5-coder:3b
Ollama connected: True
Installed models: ['qwen2.5-coder:3b-instruct']
Results directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\llm


In [3]:
import json
from urllib.request import Request, urlopen

# 1. Chọn mô hình đã cài đặt
MODEL_NAME = "qwen2.5-coder:3b-instruct"

# 2. Tạo câu hỏi thử nghiệm
prompt = "Write SQL to count all rows in the Claim table. Return only SQL."

# 3. Gửi yêu cầu tới Ollama
payload = {
    "model": MODEL_NAME,
    "prompt": prompt,
    "stream": False,
    "options": {
        "temperature": 0,
        "num_ctx": 4096
    }
}

request = Request(
    f"{OLLAMA_URL}/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urlopen(request, timeout=180) as response:
    result = json.load(response)

# 4. Hiển thị kết quả
print("Model:", result["model"])
print("Response:", result["response"].strip())
print("Total tokens:", result.get("eval_count", 0))
print("Generation time (s):", round(result.get("total_duration", 0) / 1e9, 2))

Model: qwen2.5-coder:3b-instruct
Response: ```sql
SELECT COUNT(*) AS total_rows FROM Claim;
```
Total tokens: 14
Generation time (s): 0.47


In [4]:
import pandas as pd
from rdflib import Graph, Namespace, RDF

# 1. Đọc file benchmark
BENCHMARK_FILE = DATA_DIR / "investigation" / "acme-benchmark.ttl"

benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

# 2. Khai báo namespace
QA = Namespace("http://models.data.world/benchmarks/QandA#")

# 3. Lấy danh sách câu hỏi
inquiries = sorted(
    benchmark.subjects(RDF.type, QA.Inquiry),
    key=str
)

questions = []

for i, inquiry in enumerate(inquiries, start=1):
    question = benchmark.value(inquiry, QA.prompt)

    questions.append({
        "ID": i,
        "Question": str(question)
    })

# 4. Hiển thị kết quả
questions_df = pd.DataFrame(questions)

print("Total questions:", len(questions_df))
display(questions_df)

Total questions: 44


,ID,Question
0,1,What is the loss ratio of each policy and agen...
1,2,Return all the policies we have by policy numb...
2,3,"What are the loss payment, loss reserve, expen..."
3,4,"What are the total loss, which is the sum of l..."
4,5,Return agents and the policy they have sold th...
5,6,What is the average policy size which is the t...
6,7,"What are the total loss, which is the sum of l..."
7,8,What is the total amount of premiums that a po...
8,9,"What are the loss payment, loss reserve, expen..."
9,10,Return agents and the policy they have sold th...


In [5]:
import re

# 1. Đường dẫn file SQL Schema
DDL_FILE = DATA_DIR / "DDL" / "ACME_small.ddl"

# 2. Đọc SQL Schema của tác giả
sql_schema = DDL_FILE.read_text(encoding="utf-8")

# 3. Đếm số bảng trong Schema
total_tables = len(
    re.findall(r"CREATE\s+TABLE", sql_schema, flags=re.IGNORECASE)
)

# 4. Hiển thị kết quả
print("=== SQL SCHEMA ===")
print("Total tables:", total_tables)
print("Schema length:", len(sql_schema))
print("\nSchema preview:")
print(sql_schema[:1000])

=== SQL SCHEMA ===
Total tables: 13
Schema length: 6154

Schema preview:

CREATE TABLE Claim
( 
	Claim_Identifier     int  NOT NULL ,
	Catastrophe_Identifier int  NULL ,
	Claim_Description    varchar(5000)  NULL ,
	Claims_Made_Date     datetime  NULL ,
	Company_Claim_Number varchar(20)  NULL ,
	Company_Subclaim_Number varchar(5)  NULL ,
	Insurable_Object_Identifier int  NULL ,
	Occurrence_Identifier int  NULL ,
	Entry_Into_Claims_Made_Program_Date datetime  NULL ,
	Claim_Open_Date      datetime  NULL ,
	Claim_Close_Date     datetime  NULL ,
	Claim_Reopen_Date    datetime  NULL ,
	Claim_Status_Code    varchar(5)  NULL ,
	Claim_Reported_Date  datetime  NULL ,
	 PRIMARY KEY (Claim_Identifier ASC),
	 FOREIGN KEY (Catastrophe_Identifier) REFERENCES Catastrophe(Catastrophe_Identifier),
 FOREIGN KEY (Claim_Identifier) REFERENCES Claim(Claim_Identifier),
 FOREIGN KEY (Insurable_Object_Identifier) REFERENCES Insurable_Object(Insurable_Object_Identifier),
 FOREIGN KEY (Occurrence_Identifier) REF

In [6]:
from rdflib import Graph, RDF, RDFS, OWL

# 1. Đọc OWL Ontology của tác giả
ONTOLOGY_FILE = DATA_DIR / "ontology" / "insurance.ttl"

ontology = Graph()
ontology.parse(ONTOLOGY_FILE, format="turtle")

# 2. Hàm lấy tên ngắn từ URI
def get_name(uri):
    if uri is None:
        return "Unknown"
    return str(uri).split("/")[-1].split("#")[-1]

# 3. Lấy danh sách OWL Classes
classes = sorted(
    get_name(c) for c in ontology.subjects(RDF.type, OWL.Class)
)

# 4. Lấy danh sách Object Properties và Datatype Properties
ontology_context = (
    "PREFIX in: <http://data.world/schema/insurance/>\n\n"
    "Classes:\n" + ", ".join(classes) + "\n"
)

for property_type, title in [
    (OWL.ObjectProperty, "Object Properties"),
    (OWL.DatatypeProperty, "Datatype Properties")
]:
    properties = sorted(
        ontology.subjects(RDF.type, property_type),
        key=str
    )

    ontology_context += f"\n{title}:\n"

    for prop in properties:
        domain = ontology.value(prop, RDFS.domain)
        range_ = ontology.value(prop, RDFS.range)

        ontology_context += (
            f"in:{get_name(prop)}: "
            f"{get_name(domain)} -> {get_name(range_)}\n"
        )

# 5. Hiển thị kết quả
print("=== OWL ONTOLOGY ===")
print("Total classes:", len(classes))
print("Ontology context length:", len(ontology_context))

print("\nOntology context:")
print(ontology_context)

=== OWL ONTOLOGY ===
Total classes: 11
Ontology context length: 1449

Ontology context:
PREFIX in: <http://data.world/schema/insurance/>

Classes:
Agent, Catastrophe, Claim, ExpensePayment, ExpenseReserve, LossPayment, LossReserve, Policy, PolicyCoverageDetail, PolicyHolder, Premium

Object Properties:
in:against: Claim -> PolicyCoverageDetail
in:hasCatastrophe: Claim -> Catastrophe
in:hasExpensePayment: Claim -> ExpensePayment
in:hasExpenseReserve: Claim -> ExpenseReserve
in:hasLossPayment: Claim -> LossPayment
in:hasLossReserve: Claim -> LossReserve
in:hasPolicy: PolicyCoverageDetail -> Policy
in:hasPolicyHolder: Policy -> PolicyHolder
in:hasPremiumAmount: PolicyCoverageDetail -> Premium
in:soldByAgent: Policy -> Agent

Datatype Properties:
in:agentId: Agent -> Unknown
in:catastropheName: Catastrophe -> Unknown
in:claimCloseDate: Claim -> dateTime
in:claimNumber: Claim -> Unknown
in:claimOpenDate: Claim -> dateTime
in:expensePaymentAmount: ExpensePayment -> Unknown
in:expenseReserveA

In [7]:
# 1. Xây dựng prompt cho Text-to-SQL
def make_sql_prompt(question):
    return f"""You are an expert in SQL.

Generate a DuckDB SQL query to answer the question.
Use only tables and columns from the provided database schema.
Return a single SELECT query.
Do not include explanations or Markdown.

Database Schema:
{sql_schema}

Question:
{question}

SQL Query:"""


# 2. Xây dựng prompt cho Text-to-SPARQL
def make_sparql_prompt(question):
    return f"""You are an expert in SPARQL.

Generate a SPARQL SELECT query to answer the question.
Use only classes and properties from the provided ontology.
Query the RDF graph directly. Do not use SERVICE.
Use the prefix in: for insurance classes and properties.
Return only SPARQL, without explanations or Markdown.

Ontology:
{ontology_context}

Question:
{question}

SPARQL Query:"""


# 3. Kiểm tra prompt với một câu hỏi trong benchmark
test_question = questions_df.iloc[42]["Question"]

sql_prompt = make_sql_prompt(test_question)
sparql_prompt = make_sparql_prompt(test_question)

print("=== TEST QUESTION ===")
print(test_question)

print("\n=== SQL PROMPT ===")
print("Length:", len(sql_prompt))
print(sql_prompt[-400:])

print("\n=== SPARQL PROMPT ===")
print("Length:", len(sparql_prompt))
print(sparql_prompt[-400:])

=== TEST QUESTION ===
How many claims do we have?

=== SQL PROMPT ===
Length: 6436
er) REFERENCES Policy_Amount(Policy_Amount_Identifier)
)

CREATE TABLE Catastrophe
( 
	Catastrophe_Identifier int  NOT NULL ,
	Catastrophe_Type_Code varchar(20)  NULL ,
	Catastrophe_Name     varchar(100)  NULL ,
	Industry_Catastrophe_Code varchar(20)  NULL ,
	Company_Catastrophe_Code varchar(20)  NULL ,
	 PRIMARY KEY (Catastrophe_Identifier ASC)
)

Question:
How many claims do we have?

SQL Query:

=== SPARQL PROMPT ===
Length: 1820
policyCoverageExpirationDate: PolicyCoverageDetail -> dateTime
in:policyEffectiveDate: Policy -> dateTime
in:policyExpirationDate: Policy -> dateTime
in:policyHolderId: PolicyHolder -> Unknown
in:policyNumber: Policy -> Unknown
in:premiumAmount: Premium -> Unknown
in:premiumAmountMonthly: Premium -> Unknown
in:premiumPeriod: Premium -> Unknown


Question:
How many claims do we have?

SPARQL Query:


In [8]:
import json
import re
from urllib.request import Request, urlopen

# 1. Hàm gọi Qwen thông qua Ollama
def run_llm(prompt):

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_ctx": 4096,
            "num_predict": 1024,
            "seed": 7
        }
    }

    request = Request(
        f"{OLLAMA_URL}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with urlopen(request, timeout=180) as response:
        result = json.load(response)

    # 2. Lấy câu trả lời và loại bỏ Markdown nếu có
    raw = result["response"].strip()

    match = re.search(
        r"```(?:sql|sparql)?\s*(.*?)```",
        raw,
        flags=re.IGNORECASE | re.DOTALL
    )

    query = match.group(1).strip() if match else raw

    # 3. Trả về truy vấn và thông tin thực thi
    return {
        "query": query,
        "raw": raw,
        "prompt_tokens": result.get("prompt_eval_count", 0),
        "output_tokens": result.get("eval_count", 0),
        "time_s": round(result.get("total_duration", 0) / 1e9, 2)
    }


# 4. Kiểm tra cả hai nhánh với cùng một câu hỏi
test_question = questions_df.iloc[42]["Question"]

for name, prompt in [
    ("SQL", make_sql_prompt(test_question)),
    ("SPARQL", make_sparql_prompt(test_question))
]:
    result = run_llm(prompt)

    print(f"\n=== {name} ===")
    print("Query:\n", result["query"])
    print("Prompt tokens:", result["prompt_tokens"])
    print("Output tokens:", result["output_tokens"])
    print("Time (s):", result["time_s"])


=== SQL ===
Query:
 SELECT COUNT(Claim_Identifier) FROM Claim
Prompt tokens: 1589
Output tokens: 10
Time (s): 15.56

=== SPARQL ===
Query:
 PREFIX in: <http://data.world/schema/insurance/>

SELECT (COUNT(?claim) AS ?totalClaims) WHERE {
  ?claim a in:Claim .
}
Prompt tokens: 473
Output tokens: 35
Time (s): 3.18


In [9]:
import pandas as pd

# 1. Đường dẫn lưu kết quả Text-to-SQL
SQL_OUTPUT = RESULTS_DIR / "text_to_sql_results.csv"

# 2. Đọc kết quả đã có để có thể tiếp tục khi bị gián đoạn
if SQL_OUTPUT.exists():
    old_results = pd.read_csv(SQL_OUTPUT)
    sql_runs = {int(row["ID"]): row for _, row in old_results.iterrows()}
else:
    sql_runs = {}

# 3. Sinh SQL cho từng câu hỏi
for _, row in questions_df.iterrows():
    question_id = int(row["ID"])
    question = row["Question"]

    # Bỏ qua câu hỏi đã sinh thành công
    if question_id in sql_runs and sql_runs[question_id]["Status"] == "OK":
        continue

    try:
        result = run_llm(make_sql_prompt(question))

        record = {
            "ID": question_id,
            "Question": question,
            "Generated SQL": result["query"],
            "Raw Response": result["raw"],
            "Prompt Tokens": result["prompt_tokens"],
            "Output Tokens": result["output_tokens"],
            "Time (s)": result["time_s"],
            "Status": "OK",
            "Error": ""
        }

    except Exception as e:
        record = {
            "ID": question_id,
            "Question": question,
            "Generated SQL": "",
            "Raw Response": "",
            "Prompt Tokens": 0,
            "Output Tokens": 0,
            "Time (s)": 0,
            "Status": "ERROR",
            "Error": str(e)
        }

    # 4. Lưu kết quả ngay sau mỗi câu hỏi
    sql_runs[question_id] = record

    sql_results_df = pd.DataFrame(
        sql_runs.values()
    ).sort_values("ID")

    sql_results_df.to_csv(SQL_OUTPUT, index=False)

    print(f"Completed: {question_id}/{len(questions_df)} - {record['Status']}", flush=True)

# 5. Tổng hợp kết quả
sql_results_df = pd.DataFrame(sql_runs.values()).sort_values("ID")

print("\n=== TEXT-TO-SQL RESULTS ===")
print("Total questions:", len(questions_df))
print("Generated:", (sql_results_df["Status"] == "OK").sum())
print("Errors:", (sql_results_df["Status"] == "ERROR").sum())
print("Saved to:", SQL_OUTPUT)

display(sql_results_df[["ID", "Question", "Generated SQL", "Status"]].head())

Completed: 1/44 - OK
Completed: 2/44 - OK
Completed: 3/44 - OK
Completed: 4/44 - OK
Completed: 5/44 - OK
Completed: 6/44 - OK
Completed: 7/44 - OK
Completed: 8/44 - OK
Completed: 9/44 - OK
Completed: 10/44 - OK
Completed: 11/44 - OK
Completed: 12/44 - OK
Completed: 13/44 - OK
Completed: 14/44 - OK
Completed: 15/44 - OK
Completed: 16/44 - OK
Completed: 17/44 - OK
Completed: 18/44 - OK
Completed: 19/44 - OK
Completed: 20/44 - OK
Completed: 21/44 - OK
Completed: 22/44 - OK
Completed: 23/44 - OK
Completed: 24/44 - OK
Completed: 25/44 - OK
Completed: 26/44 - OK
Completed: 27/44 - OK
Completed: 28/44 - OK
Completed: 29/44 - OK
Completed: 30/44 - OK
Completed: 31/44 - OK
Completed: 32/44 - OK
Completed: 33/44 - OK
Completed: 34/44 - OK
Completed: 35/44 - OK
Completed: 36/44 - OK
Completed: 37/44 - OK
Completed: 38/44 - OK
Completed: 39/44 - OK
Completed: 40/44 - OK
Completed: 41/44 - OK
Completed: 42/44 - OK
Completed: 43/44 - OK
Completed: 44/44 - OK

=== TEXT-TO-SQL RESULTS ===
Total questi

,ID,Question,Generated SQL,Status
0,1,What is the loss ratio of each policy and agen...,"SELECT \n p.Policy_Number,\n ap.Party_ID...",OK
1,2,Return all the policies we have by policy numb...,"SELECT Policy_Identifier, Effective_Date, Expi...",OK
2,3,"What are the loss payment, loss reserve, expen...","SELECT \n c.Claim_Number,\n p.Policy_Num...",OK
3,4,"What are the total loss, which is the sum of l...","SELECT \n c.Catastrophe_Name, \n p.Polic...",OK
4,5,Return agents and the policy they have sold th...,"SELECT \n a.Agent_Id,\n p.Policy_Number,...",OK


In [10]:
import pandas as pd

# 1. Đường dẫn lưu kết quả Text-to-SPARQL
SPARQL_OUTPUT = RESULTS_DIR / "text_to_sparql_results.csv"

# 2. Đọc kết quả đã lưu nếu có
if SPARQL_OUTPUT.exists():
    old_results = pd.read_csv(SPARQL_OUTPUT)
    sparql_runs = {
        int(row["ID"]): row
        for row in old_results.to_dict("records")
    }
else:
    sparql_runs = {}

# 3. Sinh SPARQL cho từng câu hỏi
for _, row in questions_df.iterrows():
    question_id = int(row["ID"])
    question = row["Question"]

    # Bỏ qua câu hỏi đã sinh thành công
    if question_id in sparql_runs and sparql_runs[question_id]["Status"] == "OK":
        continue

    try:
        result = run_llm(make_sparql_prompt(question))

        record = {
            "ID": question_id,
            "Question": question,
            "Generated SPARQL": result["query"],
            "Raw Response": result["raw"],
            "Prompt Tokens": result["prompt_tokens"],
            "Output Tokens": result["output_tokens"],
            "Time (s)": result["time_s"],
            "Status": "OK",
            "Error": ""
        }

    except Exception as e:
        record = {
            "ID": question_id,
            "Question": question,
            "Generated SPARQL": "",
            "Raw Response": "",
            "Prompt Tokens": 0,
            "Output Tokens": 0,
            "Time (s)": 0,
            "Status": "ERROR",
            "Error": str(e)
        }

    # 4. Lưu kết quả ngay sau mỗi câu hỏi
    sparql_runs[question_id] = record

    sparql_results_df = pd.DataFrame(
        sparql_runs.values()
    ).sort_values("ID")

    sparql_results_df.to_csv(SPARQL_OUTPUT, index=False)

    print(
        f"Completed: {question_id}/{len(questions_df)} - {record['Status']}",
        flush=True
    )

# 5. Tổng hợp kết quả
sparql_results_df = pd.DataFrame(
    sparql_runs.values()
).sort_values("ID")

print("\n=== TEXT-TO-SPARQL RESULTS ===")
print("Total questions:", len(questions_df))
print("Generated:", (sparql_results_df["Status"] == "OK").sum())
print("Errors:", (sparql_results_df["Status"] == "ERROR").sum())
print("Saved to:", SPARQL_OUTPUT)

display(
    sparql_results_df[
        ["ID", "Question", "Generated SPARQL", "Status"]
    ].head()
)

Completed: 1/44 - OK
Completed: 2/44 - OK
Completed: 3/44 - OK
Completed: 4/44 - OK
Completed: 5/44 - OK
Completed: 6/44 - OK
Completed: 7/44 - OK
Completed: 8/44 - OK
Completed: 9/44 - OK
Completed: 10/44 - OK
Completed: 11/44 - OK
Completed: 12/44 - OK
Completed: 13/44 - OK
Completed: 14/44 - OK
Completed: 15/44 - OK
Completed: 16/44 - OK
Completed: 17/44 - OK
Completed: 18/44 - OK
Completed: 19/44 - OK
Completed: 20/44 - OK
Completed: 21/44 - OK
Completed: 22/44 - OK
Completed: 23/44 - OK
Completed: 24/44 - OK
Completed: 25/44 - OK
Completed: 26/44 - OK
Completed: 27/44 - OK
Completed: 28/44 - OK
Completed: 29/44 - OK
Completed: 30/44 - OK
Completed: 31/44 - OK
Completed: 32/44 - OK
Completed: 33/44 - OK
Completed: 34/44 - OK
Completed: 35/44 - OK
Completed: 36/44 - OK
Completed: 37/44 - OK
Completed: 38/44 - OK
Completed: 39/44 - OK
Completed: 40/44 - OK
Completed: 41/44 - OK
Completed: 42/44 - OK
Completed: 43/44 - OK
Completed: 44/44 - OK

=== TEXT-TO-SPARQL RESULTS ===
Total que

,ID,Question,Generated SPARQL,Status
0,1,What is the loss ratio of each policy and agen...,PREFIX in: <http://data.world/schema/insurance...,OK
1,2,Return all the policies we have by policy numb...,PREFIX in: <http://data.world/schema/insurance...,OK
2,3,"What are the loss payment, loss reserve, expen...",PREFIX in: <http://data.world/schema/insurance...,OK
3,4,"What are the total loss, which is the sum of l...",PREFIX in: <http://data.world/schema/insurance...,OK
4,5,Return agents and the policy they have sold th...,PREFIX in: <http://data.world/schema/insurance...,OK


In [11]:
import json
import pandas as pd

# 1. Đọc kết quả của hai nhánh
sql_df = pd.read_csv(SQL_OUTPUT)
sparql_df = pd.read_csv(SPARQL_OUTPUT)

# 2. Kiểm tra số lượng và sự tương ứng của các câu hỏi
assert len(sql_df) == 44, "Text-to-SQL chưa đủ 44 câu hỏi"
assert len(sparql_df) == 44, "Text-to-SPARQL chưa đủ 44 câu hỏi"

assert set(zip(sql_df["ID"], sql_df["Question"])) == set(
    zip(sparql_df["ID"], sparql_df["Question"])
), "Hai nhánh sử dụng bộ câu hỏi khác nhau"

# 3. Thống kê kết quả sinh truy vấn
summary = []

for name, df in [
    ("Text-to-SQL", sql_df),
    ("Text-to-SPARQL", sparql_df)
]:
    summary.append({
        "Method": name,
        "Questions": len(df),
        "Generated": (df["Status"] == "OK").sum(),
        "Errors": (df["Status"] == "ERROR").sum(),
        "Prompt Tokens": df["Prompt Tokens"].sum(),
        "Output Tokens": df["Output Tokens"].sum(),
        "Total Time (s)": round(df["Time (s)"].sum(), 2),
        "Average Time (s)": round(df["Time (s)"].mean(), 2)
    })

summary_df = pd.DataFrame(summary)

# 4. Ghép truy vấn SQL và SPARQL theo từng câu hỏi
comparison_df = sql_df[
    ["ID", "Question", "Generated SQL", "Status"]
].merge(
    sparql_df[["ID", "Question", "Generated SPARQL", "Status"]],
    on=["ID", "Question"],
    suffixes=(" SQL", " SPARQL"),
    validate="one_to_one"
)

# 5. Lưu bảng thống kê và bảng so sánh
summary_df.to_csv(
    RESULTS_DIR / "generation_summary.csv",
    index=False
)

comparison_df.to_csv(
    RESULTS_DIR / "generated_queries.csv",
    index=False
)

# 6. Lưu cấu hình thực nghiệm
config = {
    "model": MODEL_NAME,
    "temperature": 0,
    "num_ctx": 4096,
    "num_predict": 1024,
    "seed": 7,
    "prompt_type": "zero-shot",
    "questions": len(questions_df),
    "sql_context": "ACME_small.ddl",
    "sparql_context": "insurance.ttl (rút gọn)"
}

with open(RESULTS_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

# 7. Hiển thị kết quả cuối cùng
print("=== TỔNG KẾT EXPERIMENT 04 ===")
print("Model:", MODEL_NAME)
print("Questions:", len(questions_df))
print("Output directory:", RESULTS_DIR)

print("\n=== GENERATION SUMMARY ===")
display(summary_df)

print("\n=== GENERATED QUERIES ===")
display(comparison_df.head())

=== TỔNG KẾT EXPERIMENT 04 ===
Model: qwen2.5-coder:3b-instruct
Questions: 44
Output directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\llm

=== GENERATION SUMMARY ===


,Method,Questions,Generated,Errors,Prompt Tokens,Output Tokens,Total Time (s),Average Time (s)
0,Text-to-SQL,44,44,0,70554,4422,106.6,2.42
1,Text-to-SPARQL,44,44,0,21450,4998,125.8,2.86



=== GENERATED QUERIES ===


,ID,Question,Generated SQL,Status SQL,Generated SPARQL,Status SPARQL
0,1,What is the loss ratio of each policy and agen...,"SELECT \n p.Policy_Number,\n ap.Party_ID...",OK,PREFIX in: <http://data.world/schema/insurance...,OK
1,2,Return all the policies we have by policy numb...,"SELECT Policy_Identifier, Effective_Date, Expi...",OK,PREFIX in: <http://data.world/schema/insurance...,OK
2,3,"What are the loss payment, loss reserve, expen...","SELECT \n c.Claim_Number,\n p.Policy_Num...",OK,PREFIX in: <http://data.world/schema/insurance...,OK
3,4,"What are the total loss, which is the sum of l...","SELECT \n c.Catastrophe_Name, \n p.Polic...",OK,PREFIX in: <http://data.world/schema/insurance...,OK
4,5,Return agents and the policy they have sold th...,"SELECT \n a.Agent_Id,\n p.Policy_Number,...",OK,PREFIX in: <http://data.world/schema/insurance...,OK
